In [1]:
# Cell 1
import sys
sys.path.append('../')
from src.compliance.alignment_engine import AlignmentEngine
from src.graph_rag.retriever         import GraphRAGRetriever
from src.graph_rag.explainer         import LLMExplainer
from src.graph_rag.report_generator  import ComplianceReportGenerator

e:\graph-rag-compliance\venv\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
# Cell 2
engine    = AlignmentEngine(env_path='../.env')
retriever = GraphRAGRetriever(engine.driver, k_hop=2)
explainer = LLMExplainer(model='claude-haiku-4-5-20251001')
generator = ComplianceReportGenerator(
    engine, retriever, explainer
)

Regulation loaded: 13 activities, 6 sequences, 4 conditions
Loading embedding model...
Embedding model loaded
LLM Explainer ready: claude-haiku-4-5-20251001


In [3]:
# Cell 3
report = generator.generate('Application_1858876732')
report.print_report()


COMPLIANCE REPORT
Mã hồ sơ     : Application_1858876732
Loại hồ sơ   : New credit
Mục đích vay : Home improvement
Số tiền vay  : 25,000 EUR
Số sự kiện   : 56
Tuân thủ     : ❌ Không
Fitness score: 0.40

Vi phạm (2):
------------------------------------------------------------

[1] HIGH — SKIP
     Điều khoản : Article 5(1)
     Mô tả      : Bước bắt buộc 'O_Sent (mail and online)' bị bỏ qua

     Graph RAG context:
     Query: O_Sent (mail and online)
     Nodes retrieved: O_Sent (mail and online), A_Complete

     Giải thích (LLM):
     # GIẢI THÍCH VI PHẠM - Application_1858876732
     ## 1. **Tóm tắt**
     Hồ sơ vay 25,000 EUR bị bỏ qua bước bắt buộc gửi thông tin tiền hợp đồng cho khách hàng (O_Sent - mail and online), dẫn đến vi phạm Điều 5(1) Chỉ thị 2008/48/EC.
     ## 2. **Lý do vi phạm**
     Theo **Điều 5(1)** của Directive 2008/48/EC, nhà cung cấp tín dụng **phải gửi thông tin tiền hợp đồng cho người vay trước khi ký kết**, bao gồm: lãi suất, phí, điều khoản thanh toán, v.v

In [5]:
# Cell debug — kiểm tra API key
import os
from dotenv import load_dotenv

load_dotenv('../.env')

key = os.getenv("ANTHROPIC_API_KEY")
if key:
    print(f"API key tìm thấy: {key[:15]}...")
else:
    print("KHÔNG tìm thấy API key")

KHÔNG tìm thấy API key


In [4]:
# Cell 4 — Test temporal violation
report2 = generator.generate('Application_35950487')
report2.print_report()


COMPLIANCE REPORT
Mã hồ sơ     : Application_35950487
Loại hồ sơ   : New credit
Mục đích vay : Existing loan takeover
Số tiền vay  : 35,000 EUR
Số sự kiện   : 63
Tuân thủ     : ❌ Không
Fitness score: 0.85

Vi phạm (1):
------------------------------------------------------------

[1] MEDIUM — TEMPORAL
     Điều khoản : Article 7(1)
     Mô tả      : Xử lý hồ sơ thiếu quá 60 ngày, vượt ngưỡng 8 ngày

     Graph RAG context:
     Query: Article 7(1) time limit deadline
     Nodes retrieved: A_Create Application, A_Submitted, A_Concept

     Giải thích (LLM):
     # BÁOCAO VI PHẠM QUY TRÌNH - Application_35950487
     ## 1. **Tóm tắt**
     Hồ sơ vay số 35950487 vượt quá thời hạn cho phép để chuyển từ trạng thái **A_Incomplete** sang **A_Validating**, với độ trễ 60 ngày thay vì tối đa 8 ngày theo tiêu chuẩn p90 của Directive 2008/48/EC.
     ## 2. **Lý do vi phạm**
     Theo **Article 7(1)** của Chỉ thị Tín dụng Tiêu dùng EU, nhà cung cấp tín dụng phải xử lý hồ sơ vay và cấp thông tin ch

In [6]:
# Cell 5 — Batch 20 case vi phạm
import json

with open('../data/processed/compliance_results.json',
          encoding='utf-8') as f:
    all_results = json.load(f)

violated_cases = [
    r['case_id'] for r in all_results
    if not r['is_compliant']
][:20]

print(f"Generating {len(violated_cases)} reports...")
reports = generator.generate_batch(
    violated_cases,
    save_path='../data/processed/explained_reports.json'
)
engine.close()

Generating 20 reports...
  5/20 generated
  10/20 generated
  15/20 generated
  20/20 generated
Đã lưu 20 report: ../data/processed/explained_reports.json


In [7]:
# Cell 6 — Thống kê kết quả batch
compliant = sum(1 for r in reports if r.is_compliant)
print(f"Tổng      : {len(reports)}")
print(f"Tuân thủ  : {compliant}")
print(f"Vi phạm   : {len(reports) - compliant}")
print(f"\nMột report mẫu có vi phạm:")
for r in reports:
    if not r.is_compliant:
        r.print_report()
        break

Tổng      : 20
Tuân thủ  : 0
Vi phạm   : 20

Một report mẫu có vi phạm:

COMPLIANCE REPORT
Mã hồ sơ     : Application_1000610355
Loại hồ sơ   : New credit
Mục đích vay : Not speficied
Số tiền vay  : 10,000 EUR
Số sự kiện   : 54
Tuân thủ     : ❌ Không
Fitness score: 0.85

Vi phạm (1):
------------------------------------------------------------

[1] MEDIUM — TEMPORAL
     Điều khoản : Article 14(1)
     Mô tả      : Rút lui sau 18 ngày, vượt quá thời hạn 14 ngày

     Graph RAG context:
     Query: Article 14(1) time limit deadline
     Nodes retrieved: A_Create Application, A_Submitted, A_Concept

     Giải thích (LLM):
     # PHÂN TÍCH VI PHẠM HỒ SƠ Application_1000610355
     ## 1. **Tóm tắt**
     Hồ sơ vay vượt quá thời hạn pháp lý để gửi lại thông tin cho khách hàng: đơn được gửi ngày 01/12/2016 nhưng rút lui trả lại sau 18 ngày (20/12/2016), vượt quá giới hạn 14 ngày theo Article 14(1).
     ## 2. **Lý do vi phạm**
     **Article 14(1)** của Directive 2008/48/EC yêu cầu nhà cung 